# Experiment 10 - Part 1: An LLM Behind a FastAPI Service, with Streaming

**Course:** HAAI302COM - Artificial Intelligence Laboratory

So far the models lived inside notebooks. A product needs them behind a **network API**: other programs send a request, the service calls the model and sends an answer back. This is an *AI microservice*. In this notebook we build one with **FastAPI**, step by step:

1. a typed `/chat` endpoint that validates its input;
2. a **streaming** endpoint that sends tokens as they are produced (Server-Sent Events), so the user sees the first words quickly;
3. a test of how `async` and `def` endpoints behave when many users arrive at the same time;
4. the automatic API documentation that FastAPI generates.

The service code is written to files by `%%writefile` cells, started in the background by `svc_utils.start_server`, and tested from the notebook with `httpx`. Notebook 02 adds a semantic cache; notebook 03 adds logging, tracing and a load test.

## 0. Setup

The service calls Gemini through the OpenAI SDK (`AsyncOpenAI`, the asynchronous client). The key is read from `.env`; it is never printed.

In [1]:
import os, time, json, warnings, statistics
warnings.filterwarnings("ignore")
import httpx
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import svc_utils

load_dotenv(find_dotenv(usecwd=True))
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 170)
print("GEMINI_API_KEY present:", bool(os.environ.get("GEMINI_API_KEY")))

GEMINI_API_KEY present: True


## 1. The request and response contract

FastAPI uses **Pydantic** models to describe the JSON that goes in and out. `Field(...)` adds rules (length limits, ranges). A request that breaks a rule is rejected automatically with status 422 *before* our code runs, so we never spend a token on bad input. `response_model` makes sure the answer has exactly the declared fields.

In [2]:
%%writefile service_v1.py
import os, time, json, asyncio
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
app = FastAPI(title="LLM microservice", version="1")
llm = AsyncOpenAI(api_key=os.environ["GEMINI_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/", max_retries=2)
MODEL = "gemini-2.5-flash-lite"


class ChatRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=2000)
    max_tokens: int = Field(200, ge=1, le=1000)
    temperature: float = Field(0.2, ge=0, le=1)


class ChatResponse(BaseModel):
    answer: str
    model: str
    latency_ms: float
    total_tokens: int

Writing service_v1.py


## 2. The endpoints: health and a blocking chat

`@app.get("/health")` is the endpoint every deployment platform calls to ask "are you alive?". `@app.post("/chat")` takes a `ChatRequest`, calls the model and returns a `ChatResponse`. The function is `async def` and uses `await`, so while the service waits for Gemini it can serve other requests. If the model provider fails we do not leak the error; we answer **502 Bad Gateway**, the standard code for a failing upstream.

The next cells use `%%writefile -a` to *append* to the same file.

In [3]:
%%writefile -a service_v1.py


@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL}


@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    t0 = time.time()
    try:
        r = await llm.chat.completions.create(model=MODEL, max_tokens=req.max_tokens, temperature=req.temperature,
                                              messages=[{"role": "user", "content": req.prompt}])
    except Exception as e:
        raise HTTPException(status_code=502, detail=f"upstream model error: {type(e).__name__}")
    return ChatResponse(answer=r.choices[0].message.content or "", model=MODEL,
                        latency_ms=round((time.time() - t0) * 1000, 1), total_tokens=r.usage.total_tokens)

Appending to service_v1.py


In [4]:
server = svc_utils.start_server("service_v1", 8101)
URL = "http://127.0.0.1:8101"
print(httpx.get(URL + "/health").json())

{'status': 'ok', 'model': 'gemini-2.5-flash-lite'}


**What to notice:** the service answers /health with the model name. It runs as a separate process (uvicorn) started from the notebook, so the notebook is now an ordinary HTTP client, exactly like a web page or a mobile app would be. The same file could run on a server with no change.

## 3. Calling the service

From the outside the service is just HTTP. We send a JSON body with `httpx.post` and measure the round-trip time ourselves; the response also reports the time the server spent on the model.

In [5]:
t = time.time()
r = httpx.post(URL + "/chat", json={"prompt": "Explain in one sentence what an API is.", "max_tokens": 80}, timeout=60)
print("status:", r.status_code, "| client round trip:", round((time.time() - t) * 1000), "ms")
print(json.dumps(r.json(), indent=1))

status: 200 | client round trip: 1750 ms
{
 "answer": "An API (Application Programming Interface) is a set of rules and specifications that allows different software applications to communicate and interact with each other.",
 "model": "gemini-2.5-flash-lite",
 "latency_ms": 1733.4,
 "total_tokens": 37
}


**What to notice:** the call returns status 200 and a JSON body with exactly the four declared fields. The client measured 1,750 ms and the server reported 1,733 ms spent on the model, so the HTTP layer on the same machine adds about 17 ms: nearly all the time is the model call. The answer is one sentence and used 37 tokens in total.

## 4. Validation: bad input never reaches the model

Three bad requests: an empty prompt, a `max_tokens` above the limit, and a missing field. FastAPI answers 422 and says which field broke which rule.

In [6]:
bad = [{"prompt": ""}, {"prompt": "hi", "max_tokens": 5000}, {"max_tokens": 10}]
for b in bad:
    r = httpx.post(URL + "/chat", json=b)
    e = r.json()["detail"][0]
    print(r.status_code, "|", b, "->", e["loc"], e["msg"])

422 | {'prompt': ''} -> ['body', 'prompt'] String should have at least 1 character
422 | {'prompt': 'hi', 'max_tokens': 5000} -> ['body', 'max_tokens'] Input should be less than or equal to 1000
422 | {'max_tokens': 10} -> ['body', 'prompt'] Field required


**What to notice:** all three bad requests got status 422 and the error names the field (the 'loc' entry) and the broken rule: an empty string, a value above the limit of 1000, and a missing required field. None of them reached Gemini, so they cost nothing. Without these rules the first user to send a 10,000 token request would decide your bill.

## 5. Streaming: send tokens as they are made

A language model produces text token by token. With the blocking endpoint the user stares at nothing until the **whole** answer is ready. A streaming endpoint sends each piece immediately. The standard browser-friendly format is **Server-Sent Events (SSE)**: a long-lived HTTP response whose body is lines of the form `data: <text>`, each followed by an empty line.

In FastAPI we return a `StreamingResponse` around an **async generator**. Each `yield` is sent to the client at once. We finish with `data: [DONE]` so that the client knows the stream ended normally.

In [7]:
%%writefile -a service_v1.py


@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    async def events():
        try:
            stream = await llm.chat.completions.create(model=MODEL, max_tokens=req.max_tokens, temperature=req.temperature, stream=True,
                                                       messages=[{"role": "user", "content": req.prompt}])
            async for chunk in stream:
                piece = chunk.choices[0].delta.content if chunk.choices else ""
                if piece:
                    yield f"data: {json.dumps({'token': piece})}\n\n"
        except Exception as e:                                   # headers are already sent, so report the error inside the stream
            yield f"data: {json.dumps({'error': type(e).__name__})}\n\n"
        yield "data: [DONE]\n\n"
    return StreamingResponse(events(), media_type="text/event-stream")

Appending to service_v1.py


In [8]:
svc_utils.stop_server(server)                      # a running server does not see new code: restart it
server = svc_utils.start_server("service_v1", 8101)
print(httpx.get(URL + "/health").json())

{'status': 'ok', 'model': 'gemini-2.5-flash-lite'}


The client side: `httpx.stream` keeps the connection open and `iter_lines` yields each line as it arrives. We print the first raw events, then measure two times for the same prompt on both endpoints: the **time to first token** (what the user feels) and the **total time**.

In [9]:
def stream_chat(prompt, max_tokens=250):
    t0, first, tokens, raw = time.time(), None, [], []
    with httpx.stream("POST", URL + "/chat/stream", json={"prompt": prompt, "max_tokens": max_tokens}, timeout=60) as r:
        for line in r.iter_lines():
            if not line.startswith("data: "):
                continue
            raw.append(line)
            if line[6:] == "[DONE]":
                break
            if first is None:
                first = time.time() - t0
            tokens.append(json.loads(line[6:]).get("token", ""))
    return "".join(tokens), first, time.time() - t0, raw

text, first, total, raw = stream_chat("Write a 120 word explanation of how a neural network learns.")
print("\n".join(raw[:3]), "\n...\n", raw[-1])
print(f"\nfirst token after {first:.2f}s, finished after {total:.2f}s, {len(text.split())} words")

data: {"token": "Neural networks learn"}
data: {"token": " through a process of iterative adjustment, much like a child learning to identify objects. Initially"}
data: {"token": ", the network is fed data with known outcomes (e.g., images of cats labeled \"cat\"). Each connection between artificial neurons has a \"weight,\" determining its influence.\n\nDuring training, the network makes a prediction. If incorrect, an"} 
...
 data: [DONE]

first token after 1.62s, finished after 2.08s, 99 words


**What to notice:** the body is a sequence of 'data:' lines, each holding a small JSON object with a piece of text, and it ends with 'data: [DONE]'. The pieces are not single tokens: Gemini sends chunks of different sizes, the first with only a few words and later ones with sentences. In this run the first event arrived after 1.6 s and the stream finished at 2.1 s with 99 words, so the first words were on screen about half a second before the end.

In [10]:
rows = []
for p in ["Write a 150 word story about a robot learning to paint.", "Explain in 150 words how the internet works.",
          "Give a 150 word summary of the causes of the French Revolution."]:
    _, first, total, _ = stream_chat(p, 300)
    t = time.time(); httpx.post(URL + "/chat", json={"prompt": p, "max_tokens": 300}, timeout=60); blocking = time.time() - t
    rows.append(dict(prompt=p[:50], stream_first_token_s=first, stream_total_s=total, blocking_wait_s=blocking))
pd.DataFrame(rows).round(2)

,prompt,stream_first_token_s,stream_total_s,blocking_wait_s
0,Write a 150 word story about a robot learning to p,0.96,1.74,1.62
1,Explain in 150 words how the internet works.,0.95,1.64,1.62
2,Give a 150 word summary of the causes of the Frenc,0.82,1.50,1.50


**What to notice:** the user sees the first words after 0.8 to 1.0 s with streaming, while the blocking endpoint makes them wait 1.5 to 1.6 s for anything; the totals are almost the same (streaming is slightly slower overall because of the extra messages). Gemini Flash-Lite is fast, so the gain here is under a second; for a 600-word answer from a slower model the blocking wait would be many seconds and the first token would still arrive in about a second. That is the reason chat products always stream.

## 6. Experiment: `async def` against `def` under load

A service is useful only if it serves many users at once. FastAPI runs `async def` endpoints on a **single event loop thread**; every `await` hands control to another request. A plain `def` endpoint is run in a **thread pool**. The classic mistake is a blocking call (here `time.sleep`, in real life a `requests.get` or a heavy computation) inside an `async def`: it freezes the whole loop.

We add three endpoints that each "work" for the same delay, in three ways, and fire 10 requests at the same time at each.

In [11]:
%%writefile -a service_v1.py


@app.get("/work-async")
async def work_async(seconds: float = 0.5):
    await asyncio.sleep(seconds)                 # cooperative wait: other requests run meanwhile
    return {"ok": True}


@app.get("/work-sync")
def work_sync(seconds: float = 0.5):
    time.sleep(seconds)                          # plain def: FastAPI runs it in a worker thread
    return {"ok": True}


@app.get("/work-blocking")
async def work_blocking(seconds: float = 0.5):
    time.sleep(seconds)                          # BUG: blocks the event loop inside async def
    return {"ok": True}

Appending to service_v1.py


In [12]:
svc_utils.stop_server(server)
server = svc_utils.start_server("service_v1", 8101)
import asyncio

# Jupyter already runs an event loop, so we can use await directly in a cell
async def hit_many(path, n=10):
    async with httpx.AsyncClient(base_url=URL, timeout=60) as c:
        t = time.time()
        await asyncio.gather(*[c.get(path) for _ in range(n)])
        return time.time() - t

rows = []
for path in ["/work-async", "/work-sync", "/work-blocking"]:
    rows.append(dict(endpoint=path, seconds_for_10_parallel_requests=await hit_many(path)))
pd.DataFrame(rows).round(2)

,endpoint,seconds_for_10_parallel_requests
0,/work-async,0.57
1,/work-sync,0.54
2,/work-blocking,5.04


**What to notice:** ten parallel requests that each need 0.5 s took 0.57 s with async def and await, and 0.54 s with plain def (FastAPI used its thread pool), which means all ten ran at the same time. With time.sleep inside an async def they took 5.04 s, ten times 0.5 s: the sleeping call froze the event loop and the requests were served one by one. The same happens with a blocking HTTP library or a heavy computation inside async def, and nothing in the code looks wrong.

## 7. Free documentation: OpenAPI

Because every endpoint is typed, FastAPI can describe the whole API in a standard machine-readable file, `/openapi.json`, and serves an interactive page at `/docs` (Swagger UI) where you can try each endpoint from the browser. Other teams can generate client code from this file.

In [13]:
spec = httpx.get(URL + "/openapi.json").json()
print("title:", spec["info"]["title"], "| paths:", list(spec["paths"]))
print("ChatRequest fields:", {k: v.get("type") for k, v in spec["components"]["schemas"]["ChatRequest"]["properties"].items()})
print("/docs status:", httpx.get(URL + "/docs").status_code)

title: LLM microservice | paths: ['/health', '/chat', '/chat/stream', '/work-async', '/work-sync', '/work-blocking']
ChatRequest fields: {'prompt': 'string', 'max_tokens': 'integer', 'temperature': 'number'}
/docs status: 200


**What to notice:** FastAPI built a description of all six endpoints and the exact types of the ChatRequest fields without us writing any documentation, and /docs returns 200 (an interactive page you can open in a browser at http://127.0.0.1:8101/docs while the server runs). Because it follows the OpenAPI standard, front-end teams can generate client code from it.

In [14]:
svc_utils.stop_server(server)
print("server stopped; log lines:", len(open("server_8101.log").read().splitlines()))

server stopped; log lines: 0


## 8. Recap and exercises

- A model becomes a service when it sits behind typed endpoints: Pydantic models validate input and output, status codes tell clients what went wrong (422 for bad input, 502 for a failing upstream).
- Streaming (SSE) makes the first words appear quickly even though the total time is the same.
- Use `async def` with `await` for I/O such as model calls; never block the event loop. A plain `def` is safe because it runs in a thread.
- `/health` and `/openapi.json` come almost for free and matter in production.

**Try yourself:** (1) add a `system` field to `ChatRequest`; (2) make `max_tokens` default to 100; (3) stop the stream when the client disconnects (check `await request.is_disconnected()`).

Next: **Notebook 02** adds a semantic cache so similar questions do not call the model again.